# Task 2 — Notebook 1: Read & join the tables

## Notebook Objective

* Read and inspect each Olist table separately, including row counts, keys, and duplicates.
* Understand what each row represents in every table.
* Aggregate `order_items` and `order_payments` before joining, since they can contain multiple rows per `order_id`.
* Build one final **ML table** as the main artifact, with **one row per order (`order_id`)**.


In [ ]:
import sys
print(sys.executable)
%pip install psycopg2-binary

In [2]:
import pandas as pd
from sqlalchemy import create_engine

# Same credentials used in Task 1's docker-compose.yml
DB_USER = "olist_user"
DB_PASS = "olist_pass"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "olist_db"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# quick connectivity check
with engine.connect() as conn:
    print("Connected OK")

Connected OK


## 1. Read every table from the database



In [3]:
table_names = [
    "orders",
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "geolocation",
    "product_category_translation",
]

tables = {}
for name in table_names:
    tables[name] = pd.read_sql_table(name, engine)
    print(tables[name].head(5))

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39 2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15

## 2. Check row counts, keys, duplicates, and what one row means



In [4]:
# Expected key column(s) per table, based on Task 1's schema (for reference/display)
expected_keys = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "product_category_translation": ["product_category_name"],
}

summary_rows = []
for name, keys in expected_keys.items():
    df = tables[name]
    n_dupe_full_row = df.duplicated().sum()  # duplicated across ALL columns, not just the key
    summary_rows.append({
        " Name ": name,
        " Key Columns ": ", ".join(keys),
        " Number of Columns ": df.shape[1],
        " Number of Rows ": df.shape[0],
        " Number of Duplicates ": n_dupe_full_row,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Name,Key Columns,Number of Columns,Number of Rows,Number of Duplicates
0,orders,order_id,8,99441,0
1,customers,customer_id,5,99441,0
2,order_items,"order_id, order_item_id",7,112650,0
3,order_payments,"order_id, payment_sequential",5,103886,0
4,order_reviews,review_id,7,98410,0
5,products,product_id,9,32951,0
6,sellers,seller_id,4,3095,0
7,geolocation,geolocation_zip_code_prefix,5,1000163,261831
8,product_category_translation,product_category_name,2,71,0


**What one row represents in each table:**

| Table | What one row represents |
|---|---|
| **orders** | One order — its core info: status, and purchase, approval, shipping, delivery, and estimated delivery dates |
| **customers** | One customer tied to a specific order (customer_id is unique per row, but the same real person can have multiple customer_ids if they ordered more than once) |
| **order_items** | One item (product) within a specific order — if an order has 3 products, it has 3 rows here, each with its own price, seller, and freight value |
| **order_payments** | One payment for a specific order — if the customer paid with more than one method or in installments, that order can have multiple rows here |
| **order_reviews** | One review the customer left for a specific order (star rating + comment) — expected to be one review per order, but Task 1 found duplicate review_ids that need attention |
| **products** | One product and its fixed attributes — category, weight, dimensions, and photo/description length — regardless of how many times it was sold |
| **sellers** | One seller and their info (city, state, zip code prefix) — not tied to a specific order |
| **geolocation** | One geographic coordinate (lat/lng) tied to a zip code prefix — the same prefix can appear in multiple rows since several coordinates are recorded per area |
| **product_category_translation** | One translation of a product category name from Portuguese to English — one row per unique category name |

In [ ]:
multi_seller_check = (
    tables["order_items"]
    .groupby("order_id")["seller_id"]
    .nunique()
)

n_multi_seller = (multi_seller_check > 1).sum()
pct_multi_seller = n_multi_seller / len(multi_seller_check) * 100

print(f"Orders with more than one distinct seller: {n_multi_seller} ({pct_multi_seller:.2f}%)")

Orders with more than one distinct seller: 59 (0.06%)


## 3. Aggregate before join
**Why aggregate before joining:**

`orders` has one row per order, but `order_items` and `order_payments` have **multiple rows per order** (one row per product, one row per payment). Joining `orders` directly with them would duplicate each order's row once per item/payment.

`GROUP BY order_id` collapses those multiple rows into one summary row per order (e.g. `n_items`, `total_price`) — matching the one-row-per-order shape needed to join with `orders`.

We keep the aggregation simple here (count, sum) since this is just to build a correct ML table. Deeper features (averages, distributions, etc.) come later in Notebook 5 (Feature Engineering), based on what we find in the EDA (Notebook 4).

In [5]:
# order_items -> one row per order
items_agg = (
    tables["order_items"]
    .groupby("order_id")
    .agg(
        n_items=("order_item_id", "count"),
        n_distinct_products=("product_id", "nunique"),
        n_distinct_sellers=("seller_id", "nunique"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
    )
    .reset_index()
)

print(items_agg.shape)
items_agg.head()

(98666, 6)


,order_id,n_items,n_distinct_products,n_distinct_sellers,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


In [8]:
# order_payments -> one row per order
payments_agg = (
    tables["order_payments"]
    .groupby("order_id")
    .agg(
        n_payments=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
    )
    .reset_index()
)

print(payments_agg.shape)
payments_agg.head()

(99440, 4)


,order_id,n_payments,total_payment_value,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


In [6]:
# order_payments -> one row per order
payments_agg = (
    tables["order_payments"]
    .groupby("order_id")
    .agg(
        n_payments=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
    )
    .reset_index()
)

print(payments_agg.shape)
payments_agg.head()

(99440, 4)


,order_id,n_payments,total_payment_value,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


## 4. Build the ML table — one row per order

We joined orders with customers to get customer information such as city and state. We also joined items_agg and payments_agg to include aggregated item and payment information.

The goal is to create an ML table where each row represents one order.

⚠️ Data Leakage: We did not use order_delivered_customer_date as a feature because it contains the actual delivery result. It will only be used later to create the label (is_late).

In [7]:
ml_table = (
    tables["orders"]
    .merge(tables["customers"], on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
)

print(ml_table.shape)
ml_table.head()

(99441, 20)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_items,n_distinct_products,n_distinct_sellers,total_price,total_freight,n_payments,total_payment_value,max_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,3.0,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,1.0,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,1.0,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,1.0,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,19.90,8.72,1.0,28.62,1.0


## 5. Sanity Checks

Before saving the ML table, we verify that:

1. The number of rows in `ml_table` is the same as the number of rows in `orders`, ensuring that the joins did not duplicate or remove orders.
2. We check for unexpected missing values caused by the `left join`, such as orders without matching items or payment records.

These checks confirm that the ML table has the expected structure and that the joins were performed correctly.


In [8]:
assert len(ml_table) == len(tables["orders"]), "Row count changed after join — check for duplication!"
print("Row count matches orders table:", len(ml_table))

assert ml_table["order_id"].duplicated().sum() == 0, "Duplicate order_id found after join!"
print("No duplicate order_id after join.")

# how many orders ended up with no items / no payments after the left join?
print("Orders with no items_agg match:", ml_table["n_items"].isna().sum())
print("Orders with no payments_agg match:", ml_table["n_payments"].isna().sum())

Row count matches orders table: 99441
No duplicate order_id after join.
Orders with no items_agg match: 775
Orders with no payments_agg match: 1


## 6. Save the Artifact

Save the `ml_table` as an artifact so that Notebook 2 can load it directly without reconnecting to the database or repeating the joins.

This makes the workflow more organized, reproducible, and efficient.


In [ ]:
import os

os.makedirs("artifacts", exist_ok=True)
ml_table.to_parquet("artifacts/ml_table.parquet", index=False)
print("Saved artifacts/ml_table.parquet ->", ml_table.shape)

Saved artifacts/ml_table.parquet -> (99441, 20)
